In [1]:
import pandas as pd
import numpy as np

# Load feature-engineered Energy dataset
energy_c13 = pd.read_csv(
    "../../Feature_engineered_datasets/energy_feature_engineered.csv"
)

print("Dataset shape:", energy_c13.shape)
print("\nColumns:")
print(energy_c13.columns.tolist())

Dataset shape: (1000, 19)

Columns:
['Timestamp', 'Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'EnergyConsumption', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status', 'Energy_Per_Occupant', 'Renewable_Energy_Ratio', 'Energy_Per_SquareFoot']


In [2]:
# Leakage-safe features
features_c13 = [
    'Temperature',
    'Humidity',
    'SquareFootage',
    'Occupancy',
    'HVACUsage',
    'LightingUsage',
    'RenewableEnergy',
    'DayOfWeek',
    'Holiday',
    'Hour',
    'Month',
    'Weekend',
    'HVAC_Status',
    'Lighting_Status'
]

target_c13 = 'EnergyConsumption'

X_c13 = energy_c13[features_c13]
y_c13 = energy_c13[target_c13]

print("Features shape:", X_c13.shape)
print("Target shape:", y_c13.shape)

print("\nSelected features:")
print(features_c13)

print("\nTarget:", target_c13)

Features shape: (1000, 14)
Target shape: (1000,)

Selected features:
['Temperature', 'Humidity', 'SquareFootage', 'Occupancy', 'HVACUsage', 'LightingUsage', 'RenewableEnergy', 'DayOfWeek', 'Holiday', 'Hour', 'Month', 'Weekend', 'HVAC_Status', 'Lighting_Status']

Target: EnergyConsumption


In [3]:
from sklearn.model_selection import train_test_split

X_train_c13, X_test_c13, y_train_c13, y_test_c13 = train_test_split(
    X_c13,
    y_c13,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train_c13.shape)
print("Testing features:", X_test_c13.shape)
print("Training target:", y_train_c13.shape)
print("Testing target:", y_test_c13.shape)

Training features: (800, 14)
Testing features: (200, 14)
Training target: (800,)
Testing target: (200,)


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Categorical and numerical features
categorical_features_c13 = [
    'HVACUsage',
    'LightingUsage',
    'DayOfWeek',
    'Holiday',
    'Weekend'
]

numerical_features_c13 = [
    'Temperature',
    'Humidity',
    'SquareFootage',
    'Occupancy',
    'RenewableEnergy',
    'Hour',
    'Month',
    'HVAC_Status',
    'Lighting_Status'
]

# Preprocessing
preprocessor_c13 = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore', drop='first'),
            categorical_features_c13
        ),
        (
            'num',
            'passthrough',
            numerical_features_c13
        )
    ]
)

# Fit only on training data
X_train_encoded_c13 = preprocessor_c13.fit_transform(X_train_c13)

# Transform test data
X_test_encoded_c13 = preprocessor_c13.transform(X_test_c13)

print("Encoded training shape:", X_train_encoded_c13.shape)
print("Encoded testing shape:", X_test_encoded_c13.shape)

Encoded training shape: (800, 19)
Encoded testing shape: (200, 19)


In [5]:
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor

# ExtraTrees base model
extra_trees_c13 = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Random Forest base model
random_forest_c13 = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("ExtraTrees model created.")
print("Random Forest model created.")

ExtraTrees model created.
Random Forest model created.


In [6]:
from sklearn.model_selection import KFold, cross_val_predict

# 5-fold cross-validation
kf_c13 = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# OOF predictions from ExtraTrees
oof_extra_trees_c13 = cross_val_predict(
    extra_trees_c13,
    X_train_encoded_c13,
    y_train_c13,
    cv=kf_c13,
    n_jobs=-1
)

# OOF predictions from Random Forest
oof_random_forest_c13 = cross_val_predict(
    random_forest_c13,
    X_train_encoded_c13,
    y_train_c13,
    cv=kf_c13,
    n_jobs=-1
)

print("ExtraTrees OOF predictions:", oof_extra_trees_c13.shape)
print("Random Forest OOF predictions:", oof_random_forest_c13.shape)

ExtraTrees OOF predictions: (800,)
Random Forest OOF predictions: (800,)


In [7]:
# Create stacking features from OOF predictions
X_meta_c13 = np.column_stack([
    oof_extra_trees_c13,
    oof_random_forest_c13
])

print("Meta-feature shape:", X_meta_c13.shape)
print("\nFirst 5 rows:")
print(X_meta_c13[:5])

Meta-feature shape: (800, 2)

First 5 rows:
[[69.51173953 69.37484169]
 [65.33963852 69.24866981]
 [72.46266481 73.40225969]
 [77.01558934 77.85644743]
 [79.53212026 79.43743695]]


In [8]:
from sklearn.linear_model import LinearRegression

# Create meta-learner
meta_learner_c13 = LinearRegression()

# Train meta-learner using OOF predictions
meta_learner_c13.fit(
    X_meta_c13,
    y_train_c13
)

print("Meta-learner trained successfully.")

print("\nMeta-learner coefficients:")
print("ExtraTrees:", meta_learner_c13.coef_[0])
print("Random Forest:", meta_learner_c13.coef_[1])
print("Intercept:", meta_learner_c13.intercept_)

Meta-learner trained successfully.

Meta-learner coefficients:
ExtraTrees: 0.14878116474497358
Random Forest: 0.806052954169523
Intercept: 3.5345217056738676


In [9]:
# Train ExtraTrees on the complete training set
extra_trees_c13.fit(
    X_train_encoded_c13,
    y_train_c13
)

# Train Random Forest on the complete training set
random_forest_c13.fit(
    X_train_encoded_c13,
    y_train_c13
)

# Generate predictions on the unseen test set
test_pred_extra_trees_c13 = extra_trees_c13.predict(
    X_test_encoded_c13
)

test_pred_random_forest_c13 = random_forest_c13.predict(
    X_test_encoded_c13
)

print("ExtraTrees test predictions:", test_pred_extra_trees_c13.shape)
print("Random Forest test predictions:", test_pred_random_forest_c13.shape)

ExtraTrees test predictions: (200,)
Random Forest test predictions: (200,)


In [10]:
# Create test meta-features
X_test_meta_c13 = np.column_stack([
    test_pred_extra_trees_c13,
    test_pred_random_forest_c13
])

# Generate final stacked predictions
y_pred_c13 = meta_learner_c13.predict(
    X_test_meta_c13
)

print("Final prediction shape:", y_pred_c13.shape)

print("\nFirst 5 actual values:")
print(y_test_c13.values[:5])

print("\nFirst 5 predicted values:")
print(y_pred_c13[:5])

Final prediction shape: (200,)

First 5 actual values:
[86.92061128 88.35160575 79.43136341 90.00918792 83.89109971]

First 5 predicted values:
[83.66797144 81.0435344  76.02342008 86.83385038 75.67753519]


In [11]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    explained_variance_score
)

# Basic metrics
mae_c13 = mean_absolute_error(y_test_c13, y_pred_c13)
mse_c13 = mean_squared_error(y_test_c13, y_pred_c13)
rmse_c13 = np.sqrt(mse_c13)
r2_c13 = r2_score(y_test_c13, y_pred_c13)

# sMAPE
smape_c13 = (
    np.mean(
        2 * np.abs(y_test_c13.values - y_pred_c13) /
        (np.abs(y_test_c13.values) + np.abs(y_pred_c13) + 1e-8)
    ) * 100
)

# NRMSE
nrmse_c13 = (
    rmse_c13 /
    (y_test_c13.max() - y_test_c13.min())
) * 100

# Additional metrics
median_ae_c13 = median_absolute_error(y_test_c13, y_pred_c13)
explained_variance_c13 = explained_variance_score(
    y_test_c13,
    y_pred_c13
)

print("C13 ExtraTrees + Random Forest Stacking")
print("-" * 50)
print(f"MAE                  : {mae_c13:.4f}")
print(f"MSE                  : {mse_c13:.4f}")
print(f"RMSE                 : {rmse_c13:.4f}")
print(f"R²                   : {r2_c13:.4f}")
print(f"sMAPE                : {smape_c13:.4f}%")
print(f"NRMSE                : {nrmse_c13:.4f}%")
print(f"Median Absolute Error: {median_ae_c13:.4f}")
print(f"Explained Variance   : {explained_variance_c13:.4f}")

C13 ExtraTrees + Random Forest Stacking
--------------------------------------------------
MAE                  : 4.4133
MSE                  : 29.8316
RMSE                 : 5.4618
R²                   : 0.5446
sMAPE                : 5.7718%
NRMSE                : 13.4314%
Median Absolute Error: 3.9337
Explained Variance   : 0.5447
